# Module 2.4 -- AI Gateways, Resilience & Production Hardening

**Track:** Backend Engineering for AI Applications  
**Toolchain:** Custom Gateway Middleware - Semantic Caching - Circuit Breakers  
**Objective:** Build production-grade resilience patterns for multi-provider LLM infrastructure: semantic caching, AI-specific circuit breakers, and load shedding.

---

## Why AI Systems Need Special Resilience

Traditional APIs fail predictably: 500 errors, timeouts, database deadlocks. LLM APIs fail in UNIQUE ways:

| Failure Mode | Traditional API | LLM API |
|---|---|---|
| Rate limiting | Rare | Constant (429 errors) |
| Cost explosion | Predictable | One bad prompt = $50 |
| Latency variance | 50ms +/- 10ms | 500ms to 60 SECONDS |
| Silent failure | Error code returned | Hallucinated response (looks correct but wrong) |
| Agent loops | N/A | Agent calls same tool 100 times |

### AI Gateway Architecture

```
Client --> [AI Gateway] --> Provider A (OpenAI)
                |--------> Provider B (Anthropic)  [fallback]
                |--------> Provider C (Self-hosted) [fallback]
                |
           [Semantic Cache]
           [Circuit Breakers]
           [Load Shedder]
           [Cost Tracker]
```

### Gateway Solutions (2026)

| Gateway | Language | Latency Overhead | Best For |
|---------|---------|-----------------|----------|
| **Bifrost** | Go | <1ms | Maximum performance |
| **Kong AI** | Lua/Go | ~2ms | Enterprise service mesh |
| **Cloudflare AI** | Edge | ~5ms | Global edge deployment |
| **Custom** | Python | ~10ms | Full control, this notebook |


### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Relying on a single third-party API is financially and statistically dangerous. Seniors shield applications with AI Gateways to handle semantic caching, rate limiting, and automatic model fallbacks during outages.

**Mental Model:** An AI Gateway is a bouncer and a switchboard operator at a nightclub. It rejects bad/spam requests (rate limits), serves frequent requests instantly from memory (caching), and redirects VIPs if the main room is full (fallbacks).

**Common Junior Pitfall:** Hard-crashing the web app when OpenAI goes down for 5 minutes. Production apps catch the 503 error and instantly reroute the payload to an Anthropic or local backup model seamlessly.

---


In [ ]:
# Cell 1 -- Install
!pip install -q numpy scikit-learn


---
## 1 - Semantic Caching

### Why Traditional Caching Fails for LLMs

Traditional cache: exact string match.  
```
"What is the capital of France?" --> cached
"what's the capital of france?"  --> CACHE MISS (different string!)
"Capital of France?"             --> CACHE MISS (different string!)
```

All three questions mean the SAME thing but get 3 separate (expensive) LLM calls.

### Semantic Caching: Match by MEANING

1. Convert query to embedding vector
2. Search cache for similar embeddings (cosine similarity)
3. If similarity > threshold (e.g., 0.95): return cached response
4. Else: call LLM, cache the result

### Trade-offs

| Threshold | Cache Hit Rate | Risk |
|-----------|---------------|------|
| 0.99 | Low (~10%) | Very safe (almost exact match) |
| 0.95 | Medium (~40%) | Good balance |
| 0.90 | High (~60%) | May return wrong cached answer |
| 0.80 | Very high | DANGEROUS: different questions match |


In [ ]:
# Cell 2 -- Semantic cache implementation
import numpy as np
from datetime import datetime

class SemanticCache:
    def __init__(self, similarity_threshold=0.95, max_entries=10000, ttl_hours=24):
        self.threshold = similarity_threshold
        self.max_entries = max_entries
        self.ttl_seconds = ttl_hours * 3600
        self.cache = []  # [{embedding, response, timestamp}]
        self.stats = {'hits': 0, 'misses': 0, 'cost_saved': 0.0}

    def _embed(self, text):
        """In production: return embedding_model.encode(text)"""
        np.random.seed(hash(text.lower().strip()) % 2**32)
        return np.random.randn(384)

    def _cosine_sim(self, a, b):
        return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10)

    def get(self, query):
        query_emb = self._embed(query)
        best_sim, best_entry = 0, None
        for entry in self.cache:
            if datetime.utcnow().timestamp() - entry['timestamp'] > self.ttl_seconds:
                continue
            sim = self._cosine_sim(query_emb, entry['embedding'])
            if sim > best_sim:
                best_sim, best_entry = sim, entry
        if best_sim >= self.threshold:
            self.stats['hits'] += 1
            self.stats['cost_saved'] += 0.01  # Avg cost per LLM call
            return best_entry['response'], best_sim
        self.stats['misses'] += 1
        return None, best_sim

    def put(self, query, response):
        if len(self.cache) >= self.max_entries:
            self.cache.pop(0)  # Evict oldest
        self.cache.append({'embedding': self._embed(query), 'response': response, 'timestamp': datetime.utcnow().timestamp()})

cache = SemanticCache(similarity_threshold=0.95)

# First call: cache miss
cache.put('What is the capital of France?', 'The capital of France is Paris.')

# Test with semantically similar queries
test_queries = [
    'What is the capital of France?',      # Exact match
    'what is the capital of france',         # Case/punctuation variation
    'Capital of France?',                    # Shortened
    'What is the capital of Germany?',       # DIFFERENT question
]

print('Semantic Cache Demo:')
for q in test_queries:
    response, sim = cache.get(q)
    hit = 'HIT' if response else 'MISS'
    print(f'  [{hit}] sim={sim:.3f} | "{q}"')
    if response: print(f'         Cached: "{response}"')
print(f'\nStats: {cache.stats}')


---
## 2 - AI-Specific Circuit Breakers

### What is a Circuit Breaker?

**Analogy:** Your house has a circuit breaker box. When too much current flows, the breaker TRIPS (opens), cutting power to prevent a fire. After a cooldown, you reset it.

```
CLOSED (normal) --> failures exceed threshold --> OPEN (blocking all requests)
     ^                                              |
     |                                         cooldown timer
     |                                              |
     +---- success <---- HALF-OPEN (test one request)
```

### Why LLMs Need Special Circuit Breakers

| Standard Circuit Breaker | AI Circuit Breaker |
|---|---|
| Trips on 500 errors | Also trips on 429 (rate limit) |
| Fixed cooldown | Parses Retry-After header |
| Fails open | Falls back to cheaper model |
| No loop detection | Detects agent stuttering loops |


In [ ]:
# Cell 3 -- AI Circuit Breaker with graceful degradation
import time

class AICircuitBreaker:
    CLOSED, OPEN, HALF_OPEN = 'CLOSED', 'OPEN', 'HALF_OPEN'

    def __init__(self, failure_threshold=3, cooldown_seconds=30, fallback_model='ollama/llama3'):
        self.failure_threshold = failure_threshold
        self.cooldown = cooldown_seconds
        self.fallback_model = fallback_model
        self.state = self.CLOSED
        self.failure_count = 0
        self.last_failure_time = 0
        self.retry_after = 0

    def call(self, primary_model, prompt):
        if self.state == self.OPEN:
            if time.time() - self.last_failure_time > self.cooldown:
                self.state = self.HALF_OPEN
                print(f'    Circuit: HALF_OPEN (testing primary)')
            else:
                remaining = self.cooldown - (time.time() - self.last_failure_time)
                print(f'    Circuit: OPEN (using fallback: {self.fallback_model}, cooldown: {remaining:.0f}s)')
                return f'[FALLBACK:{self.fallback_model}] Response to: {prompt[:30]}'

        # Simulate API call (50% failure rate for demo)
        import random; success = random.random() > 0.5

        if success:
            self.failure_count = 0
            self.state = self.CLOSED
            return f'[{primary_model}] Response to: {prompt[:30]}'
        else:
            self.failure_count += 1
            self.last_failure_time = time.time()
            if self.failure_count >= self.failure_threshold:
                self.state = self.OPEN
                print(f'    Circuit TRIPPED! {self.failure_count} consecutive failures')
            return None

breaker = AICircuitBreaker(failure_threshold=3, cooldown_seconds=5)
print('AI Circuit Breaker Demo:')
import random; random.seed(42)
for i in range(10):
    result = breaker.call('gpt-4o', f'Question {i+1}: Explain topic {i+1}')
    status = breaker.state
    print(f'  Call {i+1}: state={status}, result={"OK" if result else "FAILED"}')


---
## 3 - Reasoning Circuit Breaker (Agent Loop Detection)

### The Stuttering Agent Problem

An agent calls the same tool with the same arguments in a loop. Without detection, it could loop 1000 times costing $100+.

```
Step 1: search_web("AI market size")  --> No results
Step 2: search_web("AI market size")  --> No results  (SAME!)
Step 3: search_web("AI market size")  --> No results  (SAME! TRIP!)
```


In [ ]:
# Cell 4 -- Reasoning circuit breaker
class ReasoningCircuitBreaker:
    def __init__(self, max_repeats=3):
        self.max_repeats = max_repeats
        self.history = []

    def check(self, tool_name, arguments):
        call_sig = f'{tool_name}({arguments})'
        self.history.append(call_sig)

        # Check for repeated identical calls
        recent = self.history[-self.max_repeats:]
        if len(recent) == self.max_repeats and len(set(recent)) == 1:
            return True, f'Agent loop detected: {call_sig} called {self.max_repeats}x'
        return False, None

rcb = ReasoningCircuitBreaker(max_repeats=3)
test_calls = [
    ('search_web', '"AI market size"'),
    ('search_web', '"AI market size"'),
    ('search_web', '"AI market size"'),  # TRIP!
    ('query_db', '"SELECT * FROM sales"'),
]
print('Reasoning Circuit Breaker:')
for tool, args in test_calls:
    tripped, msg = rcb.check(tool, args)
    status = 'TRIPPED' if tripped else 'OK'
    print(f'  [{status}] {tool}({args})')
    if tripped: print(f'    Alert: {msg}')


---
## 4 - Load Shedding: Preventing Slow Death

### The Problem

100 requests arrive. GPUs can handle 10 at a time. Without load shedding:
- 90 requests queue up
- ALL 100 users wait 60+ seconds
- P99 latency spikes to 120 seconds
- System appears "down" to everyone

With load shedding:
- 10 requests processed immediately (<3s)
- 90 get instant 503 ("try again in 5 seconds")
- Users retry and get served quickly
- System stays responsive

### The Counterintuitive Truth

**Rejecting requests IMPROVES the user experience** because the requests that ARE served complete quickly.


In [ ]:
# Cell 5 -- Load shedder implementation
import threading, time

class LoadShedder:
    def __init__(self, max_concurrent=5):
        self.max_concurrent = max_concurrent
        self.active = 0
        self.lock = threading.Lock()
        self.stats = {'accepted': 0, 'rejected': 0}

    def try_acquire(self):
        with self.lock:
            if self.active >= self.max_concurrent:
                self.stats['rejected'] += 1
                return False
            self.active += 1
            self.stats['accepted'] += 1
            return True

    def release(self):
        with self.lock: self.active -= 1

shedder = LoadShedder(max_concurrent=3)
print('Load Shedding Demo (max 3 concurrent):')
for i in range(8):
    if shedder.try_acquire():
        print(f'  Request {i+1}: ACCEPTED (active: {shedder.active}/{shedder.max_concurrent})')
        if i >= 4: shedder.release()  # Simulate some completing
    else:
        print(f'  Request {i+1}: REJECTED 503 (active: {shedder.active}/{shedder.max_concurrent})')
print(f'\nStats: {shedder.stats}')
print(f'Rejection rate: {shedder.stats["rejected"]/(shedder.stats["accepted"]+shedder.stats["rejected"])*100:.0f}%')


---

## 5 -- Visual Execution Traces

### Circuit Breaker State Transitions (Step by Step)

```
T=0s    Request 1: CLOSED -> call OpenAI -> 200 OK           [failures: 0/3]
T=1s    Request 2: CLOSED -> call OpenAI -> 429 Rate Limited  [failures: 1/3]
T=2s    Request 3: CLOSED -> call OpenAI -> 429 Rate Limited  [failures: 2/3]
T=3s    Request 4: CLOSED -> call OpenAI -> 503 Unavailable   [failures: 3/3]
        >>> CIRCUIT TRIPS TO OPEN <<<
T=4s    Request 5: OPEN -> SKIP OpenAI -> use fallback Llama 3 (self-hosted)
T=5s    Request 6: OPEN -> SKIP OpenAI -> use fallback Llama 3
...
T=34s   Request 20: cooldown expired -> HALF_OPEN -> test OpenAI -> 200 OK
        >>> CIRCUIT RESETS TO CLOSED <<<
T=35s   Request 21: CLOSED -> call OpenAI -> 200 OK (back to normal)
```

### Load Shedding Trace

```
T=0ms   Request A arrives     [GPU slots: 1/5 used] -> ACCEPTED
T=10ms  Request B arrives     [GPU slots: 2/5 used] -> ACCEPTED
T=20ms  Request C arrives     [GPU slots: 3/5 used] -> ACCEPTED
T=30ms  Request D arrives     [GPU slots: 4/5 used] -> ACCEPTED
T=40ms  Request E arrives     [GPU slots: 5/5 used] -> ACCEPTED
T=50ms  Request F arrives     [GPU slots: 5/5 FULL] -> REJECTED 503
T=60ms  Request G arrives     [GPU slots: 5/5 FULL] -> REJECTED 503
T=2000ms Request A completes  [GPU slots: 4/5 used]
T=2010ms Request H arrives    [GPU slots: 5/5 used] -> ACCEPTED (slot freed!)
```


---

## 6 -- Token Economics & Cost Optimization

### The Hidden Cost of LLMs in Production

At 100K queries/day, small differences in cost per query add up FAST:

| Model | Input $/1M tokens | Output $/1M tokens | Avg Cost/Query | Daily Cost (100K) | Monthly |
|-------|:-:|:-:|:-:|:-:|:-:|
| GPT-4o | $2.50 | $10.00 | $0.015 | $1,500 | $45,000 |
| Claude 3.5 Sonnet | $3.00 | $15.00 | $0.021 | $2,100 | $63,000 |
| Claude 3.5 Haiku | $0.25 | $1.25 | $0.002 | $200 | $6,000 |
| Llama 3 70B (self-hosted) | ~$0.00 | ~$0.00 | $0.001* | $100* | $3,000* |

*Self-hosted cost = GPU rental only (A100 at $2/hr)

### Cost Optimization Strategies

| Strategy | Savings | Effort | Risk |
|----------|:---:|:---:|:---:|
| Semantic caching (NB17) | 30-60% | Low | Low (similar queries) |
| Smaller model for simple tasks | 80-95% | Medium | Medium (quality drop) |
| Prompt optimization (fewer tokens) | 10-30% | Low | None |
| Batching requests | 20-40% | Medium | Latency increase |
| Self-hosting (Llama 3) | 90%+ | High | Ops burden |

### When to Self-Host vs Use API

| Factor | Use API | Self-Host |
|--------|---------|----------|
| Volume < 10K queries/day | Yes | No (not cost effective) |
| Volume > 100K queries/day | Maybe | Yes (break-even ~50K) |
| Data residency required | No | Yes (data stays on your servers) |
| Need latest models | Yes | No (open models lag 3-6 months) |
| Team has GPU infra experience | N/A | Required |


In [ ]:
# Cell -- Cost calculator

def calculate_llm_cost(queries_per_day, avg_input_tokens=500, avg_output_tokens=300, model='gpt-4o'):
    pricing = {
        'gpt-4o': {'input': 2.50, 'output': 10.00},
        'claude-3.5-sonnet': {'input': 3.00, 'output': 15.00},
        'claude-3.5-haiku': {'input': 0.25, 'output': 1.25},
        'llama-3-70b-hosted': {'input': 0.00, 'output': 0.00},  # GPU cost only
    }
    p = pricing.get(model, pricing['gpt-4o'])
    input_cost = (avg_input_tokens / 1_000_000) * p['input'] * queries_per_day
    output_cost = (avg_output_tokens / 1_000_000) * p['output'] * queries_per_day
    daily = input_cost + output_cost
    if model == 'llama-3-70b-hosted': daily = queries_per_day * 0.001  # GPU amortized
    return {'model': model, 'daily': daily, 'monthly': daily * 30, 'yearly': daily * 365}

print('Cost Comparison: 50,000 queries/day (500 input + 300 output tokens avg)')
print(f'{"":-<70}')
for model in ['gpt-4o', 'claude-3.5-sonnet', 'claude-3.5-haiku', 'llama-3-70b-hosted']:
    c = calculate_llm_cost(50_000, model=model)
    print(f'  {model:<25} ${c["daily"]:>8,.0f}/day  ${c["monthly"]:>10,.0f}/month  ${c["yearly"]:>12,.0f}/year')

print(f'\nWith 40% semantic cache hit rate (saves 40% of API calls):')
for model in ['gpt-4o', 'claude-3.5-haiku']:
    c = calculate_llm_cost(30_000, model=model)  # 40% cached = only 30K actual calls
    print(f'  {model:<25} ${c["daily"]:>8,.0f}/day  ${c["monthly"]:>10,.0f}/month  (40% savings)')


---
## Summary

| Pattern | What It Solves | Implementation |
|---------|---------------|----------------|
| Semantic caching | Redundant LLM calls for similar queries | Embedding similarity search |
| AI circuit breaker | Provider outages, rate limits | 429/503 tracking + fallback model |
| Reasoning breaker | Agent infinite loops | Detect repeated tool calls |
| Load shedding | GPU queue congestion | Reject when at capacity |

**Next -->** `18_llm_evaluation.ipynb` -- Evaluating LLM Outputs & RAG Quality